# Proyecto — Data Stream Processor

## Contexto (extremadamente importante): 

Uno de las mayores virtudes de un repositorio en github es la poder volver sobre los cambios hechos. Uno puede volver sbre algún `commit` he iniciar el proceso desde ese punto. 

En otro ejemplo parecido, al crear una lista en python, dicha lista se modifica al agregar o borrar elementos y regresar a un estado anterior de la lista no es tan sencillo. La idea de este proyecto es poder emular dicho proceso y realizar una especie de lista con memoria para poder llevar algunos registros de manera adecuada.


### Objetivo

Construya un pequeño sistema para recibir y procesar registros de datos utilizando las clases `ArrayStack` y `ArrayQueue` proporcionadas por el curso. La idea central es poder llevar la información en orden para llevar los cambios de los registros de manera adecuada.

En el método `__init__` debe aparecer tres atributos:
- **Queue:** registros que han llegado pero todavía no han sido procesados.
- **Stack:** historial de cambios realizados, para poder deshacer los cambios más recientes.
- **Lista:** como se encuentran los registros actualmente

Las clases `ArrayStack` y `ArrayQueue` ya están implementadas. **No debe implementarlas nuevamente ni modificarlas.**

## 1. Registro de datos

Cada registro es una tupla de tres elementos:

```python
(sensor, variable, value)
```

Ejemplos:

```python
("S01", "temperature", 23.5)
("S02", "temperature", 25.1)
("S01", "humidity", 61.2)
```

Una combinación única `(sensor, variable)` identifica un dato dentro del estado actual.

## 2. Clase `DataProcessor`

Implemente:

```python
class DataProcessor:
    ...
```

Debe utilizar:

- un `ArrayQueue` para los registros pendientes;
- un `ArrayStack` para el historial de cambios;
- un `list` para mantener el estado actual.

### Restricción

No sustituya `ArrayQueue` o `ArrayStack` por `list`, `collections.deque` u otra estructura para realizar las funciones que corresponden a la Queue o al Stack.

La clase `DataProcessor` no debe imprimir resultados. Los métodos deben devolver los valores especificados. Las impresiones utilizadas para demostrar el funcionamiento deben realizarse en las celdas de prueba.

In [30]:
from goodrich.ch06.array_queue import ArrayQueue
from goodrich.ch06.array_stack import ArrayStack
from goodrich.exceptions import Empty

In [31]:
class DataProcessor:
    def __init__(self):
        self._queue = ArrayQueue()      # registros pendientes
        self._stack = ArrayStack()      # historial para deshacer
        self._state = []                # estado actual: [(sensor, variable, value), ...]


## 3. `add(record)`

Agrega un registro a la `ArrayQueue`.

```python
processor.add(("S01", "temperature", 23.5))
```

Requisitos:

- agrega el registro a la cola;
- **no procesa** el registro;
- conserva el orden de llegada.

No se requiere un valor de retorno.

Debe rechazar registros que no tengan exactamente tres componentes o cuyo `value` no sea numérico. El tipo concreto de excepción para estos errores puede ser elegido por el estudiante, pero debe documentarse y utilizarse consistentemente.

In [32]:
def add(self, record):
    if len(record) != 3:
        raise ValueError("El registro debe tener exactamente 3 elementos")

    sensor = record[0]
    variable = record[1]
    value = record[2]

    if type(value) != int and type(value) != float:
        raise ValueError("El value debe ser numérico")

    self._queue.enqueue(record)


DataProcessor.add = add #Pegaralo a la clase 


## 4. `process_next()`

Procesa el siguiente registro pendiente.

Debe:

1. obtener el siguiente registro de la Queue;
2. procesarlo;
3. actualizar el estado actual;
4. guardar en el Stack la información necesaria para poder deshacer exactamente ese cambio.

### FIFO

Si se ejecuta:

```python
add(A)
add(B)
add(C)
```

las llamadas sucesivas a `process_next()` deben devolver/procesar `A`, luego `B` y luego `C`.

### Actualización

Si se procesa:

```python
("S01", "temperature", 23.5)
```

en el estado se debe reflejar:

```python
('S01', 'temperature', 23.5)
```

Si después se procesa `("S01", "temperature", 27.0)`, el valor actual debe ser `27.0`.

### Historial

Se debe agregar al `Stack` respectivo

### Retorno

Debe devolver el registro que acaba de ser procesado.

### Queue vacía

Si no hay registros pendientes, debe producir `Empty, el error creado en el repositorio `Goodrich`

In [33]:
def process_next(self):
    record = self._queue.dequeue()   # si está vacía, esto lanza Empty solo
    sensor = record[0]
    variable = record[1]

    # Buscar si ya existe ese sensor y variable en el estado
    indice_encontrado = None
    for i in range(len(self._state)):
        if self._state[i][0] == sensor and self._state[i][1] == variable:
            indice_encontrado = i

    if indice_encontrado is None:
        # Es nuevo, lo agrego al final del estado
        self._state.append(record)
        self._stack.push(("nuevo", sensor, variable))
    else:
        # Ya existía, guardo cómo estaba antes para poder deshacer
        registro_viejo = self._state[indice_encontrado]
        self._state[indice_encontrado] = record
        self._stack.push(("actualizacion", registro_viejo))

    return record
DataProcessor.process_next= process_next #Pegaralo a la clase 

## 5. `undo()`

Deshace el último cambio realizado mediante `process_next()`.

Ejemplo:

```text
20 → 25 → 30
```

Después de un `undo()`:

```text
20 → 25
```

Después de otro:

```text
20
```

Los cambios deben deshacerse en orden LIFO.

### Dato creado por primera vez

Suponga que inicialmente no existe `('S01', 'temperature', x)`, después de procesar:

```python
("S01", "temperature", 23.5)
```

el dato existe. Si se ejecuta `undo()`, debe volver a **no existir**.

### Historial vacío

Si no hay cambios que deshacer, debe producir `Empty`.

No se requiere un valor de retorno.

In [34]:
def undo(self):
    accion = self._stack.pop()   # si está vacío, esto lanza Empty solo
    tipo_de_accion = accion[0]

    if tipo_de_accion == "nuevo":
        sensor = accion[1]
        variable = accion[2]

        indice_encontrado = None
        for i in range(len(self._state)):
            if self._state[i][0] == sensor and self._state[i][1] == variable:
                indice_encontrado = i

        del self._state[indice_encontrado]

    else:
        registro_viejo = accion[1]
        sensor = registro_viejo[0]
        variable = registro_viejo[1]

        indice_encontrado = None
        for i in range(len(self._state)):
            if self._state[i][0] == sensor and self._state[i][1] == variable:
                indice_encontrado = i

        self._state[indice_encontrado] = registro_viejo
        
DataProcessor.undo= undo #Pegaralo a la clase 

## 6. `pending()`

Devuelve el número de registros que todavía esperan ser procesados.

Por ejemplo, después de:

```python
add(A)
add(B)
add(C)
```

`pending()` debe devolver `3`. Después de `process_next()`, debe devolver `2`.

In [35]:
def pending(self):
    return len(self._queue)

DataProcessor.pending= pending #Pegaralo a la clase 

## 7. `current_value(sensor, variable)`

Devuelve el valor actual asociado con una combinación de sensor y variable.

Ejemplo:

```python
current_value("S01", "temperature")
```

puede devolver `23.5`.

Si nunca se ha procesado un registro para esa combinación, debe producir `KeyError`.

In [36]:
def current_value(self, sensor, variable):
    for i in self._state:
        if i[0] == sensor and i[1] == variable: 
            return i[2]
    raise KeyError("No existe ningun registro con esa combinación")

DataProcessor.current_value= current_value #Pegaralo a la clase

# 8. Ejemplo completo

Considere:

```python
A = ("S01", "temperature", 20)
B = ("S01", "temperature", 25)
C = ("S01", "humidity", 60)
```

Después de `add(A)`, `add(B)`, `add(C)`, la Queue contiene `A → B → C`.

Después de procesar A, el estado contiene:

```text
S01 / temperature → 20
```

Después de procesar B:

```text
S01 / temperature → 25
```

Después de procesar C:

```text
S01 / temperature → 25
S01 / humidity    → 60
```

Un `undo()` elimina el efecto de C. Otro `undo()` elimina el efecto de B. Otro `undo()` elimina el efecto de A y el estado vuelve a estar vacío.

In [37]:
procesador = DataProcessor() #Creo la instancia

A = ("S01", "temperature", 20) #Defino las tuplas con los valores que me exigen
B = ("S01", "temperature", 25)
C = ("S01", "humidity", 60)

procesador.add(A)
procesador.add(B) # Agrego 3 tuplas a la clase 
procesador.add(C)

print("Pendientes iniciales:", procesador.pending()) #Revisar cuantos registros fueron agregados al procesador 

#Se procesa cada tupla en el orden en que le va entrando 
pro_A = procesador.process_next() 
print("Se proceso A", pro_A)

pro_B = procesador.process_next()
print("Se proceso B", pro_B)

pro_C = procesador.process_next()
print("Se proceso C", pro_C)

#Vamos a eliminar el efecto de cada tupla
procesador.undo()
print("Estado después de deshacer C:", procesador._state)
procesador.undo()
print("Estado después de deshacer B:", procesador._state)
procesador.undo()
print("Estado después de deshacer A:", procesador._state)

print("Pendientes finales:", procesador.pending()) #Revisar si se realizaron los rgistros 

Pendientes iniciales: 3
Se proceso A ('S01', 'temperature', 20)
Se proceso B ('S01', 'temperature', 25)
Se proceso C ('S01', 'humidity', 60)
Estado después de deshacer C: [('S01', 'temperature', 25)]
Estado después de deshacer B: [('S01', 'temperature', 20)]
Estado después de deshacer A: []
Pendientes finales: 0


# 9. Pruebas obligatorias

Incluya pruebas para, como mínimo:

1. `pending()` sobre un procesador vacío.
2. Agregar un registro.
3. Agregar varios registros.
4. Verificar procesamiento FIFO.
5. Procesar un registro.
6. Procesar varios registros.
7. Actualizar una variable existente.
8. Consultar el valor actual.
9. Realizar un `undo()`.
10. Realizar varios `undo()` consecutivos.
11. Procesar cuando la Queue está vacía.
12. Hacer `undo()` cuando el historial está vacío.
13. Deshacer la creación de un dato que antes no existía.
14. Hacer varios cambios sobre la misma variable.
15. Agregar un registro con formato incorrecto.
16. Agregar un registro cuyo valor no sea numérico.
17. Consultar un sensor/variable que nunca haya sido procesado.

In [38]:
procesador_1 = DataProcessor() # Creo la instancia
# 1
print("Estado del procesador",procesador_1.pending()) # Se verifica que el procesador este vacio

# 2 Agregamos un registro
J = ("S02", "Motion", 40)
procesador_1.add(J)
print("Verificar si se agrego",procesador_1.pending())

# 3 Agregar varios registros
registros_multiples = [("S02", "Motion", 50),("S03", "Temperature", 22),("S03", "Temperature", 24)]

for i in registros_multiples:
    procesador_1.add(i)

print("Pendientes después de agregar varios:", procesador_1.pending()-1)

#4 y 5 y 6  Verificar FIFO y procesar uno y varios registros 
print(procesador_1.process_next()) #El primero que entra el primero qeu sale
print(procesador_1.process_next()) # y asi sucesivamente
print(procesador_1.process_next()) 
print(procesador_1.process_next()) #El ultimo en entrar el ultimo en salir 

#7 y 8 Actualizar y revisar el nuevo valor 
J_actualizado = ("S02", "Motion", 85)
procesador_1.add(J_actualizado)
procesador_1.process_next()
print("Valor actualizado de J :", procesador_1.current_value("S02", "Motion"))

# 9 y 10. Realizar un undo() y varios undo() consecutivos
procesador_1.undo()
procesador_1.undo()

# 11. Procesar cuando la Queue está vacía (capturando el error esperado)
try:
    procesador_1.process_next()
except Exception as e:
    print("11. Error capturado correctamente por cola vacía:", type(e))

# 12. Hacer undo() cuando el historial está vacío (capturando el error esperado)
try:
    procesador_1.undo()
except Exception as e:
    print("12. Error capturado correctamente por historial vacío:", type(e))

# 13. Deshacer la creación de un dato que antes no existía
N = ("S04", "Humidity", 50)
procesador_1.add(N)
procesador_1.process_next()
procesador_1.undo()

# 14. Hacer varios cambios sobre la misma variable
procesador_1.add(("S05", "Battery", 10))
procesador_1.process_next()
procesador_1.add(("S05", "Battery", 20))
procesador_1.process_next()

# 15. Agregar un registro con formato incorrecto
try:
    procesador_1.add("FormatoIncorrecto")
except Exception as e:
    print("15. Error capturado por formato:", type(e))

# 16. Agregar un registro cuyo valor no sea numérico
try:
    procesador_1.add(("S05", "Battery", "chefcito"))
except Exception as e:
    print("16. Error capturado por valor no numérico:", type(e))

# 17. Consultar un sensor/variable que nunca haya sido procesado
try:
    procesador_1.current_value("S99", "Inexistente")
except KeyError:
    print("17. KeyError capturado correctamente al consultar variable inexistente.")


Estado del procesador 0
Verificar si se agrego 1
Pendientes después de agregar varios: 3
('S02', 'Motion', 40)
('S02', 'Motion', 50)
('S03', 'Temperature', 22)
('S03', 'Temperature', 24)
Valor actualizado de J : 85
11. Error capturado correctamente por cola vacía: <class 'goodrich.exceptions.Empty'>
15. Error capturado por formato: <class 'ValueError'>
16. Error capturado por valor no numérico: <class 'ValueError'>
17. KeyError capturado correctamente al consultar variable inexistente.


# 10. Análisis de complejidad

Explique la complejidad temporal de:

- `add`
- `process_next`
- `undo`
- `pending`
- `current_value`

Justifique qué operaciones determinan cada complejidad e indique qué estructuras auxiliares utiliza el sistema.

10. Análisis de complejidad

Complejidad temporal de cada método:

- add: Es O(1) constante. Solo revisa que el registro tenga tres cosas, que el valor sea un número y lo tira al final de la fila. Como no importa cuántos datos haya guardados, siempre se demora lo mismo.

- process_next: Es O(N) lineal. Saca el primer registro de la fila y luego revisa con un ciclo la lista de estado actual para ver si ese sensor y variable ya existían. El tiempo depende de cuántas variables únicas tengamos almacenadas.

- undo: Es O(N) lineal. Saca la última acción del historial y usa un ciclo para buscar en la lista de estado la posición exacta que debe modificar o borrar.

- pending: Es O(1) constante. Solo usa la función len() para contar de inmediato cuántos elementos esperan en la fila.

- current_value: Es O(N) lineal. Busca elemento por elemento en la lista de estado hasta encontrar la combinación correcta del sensor y la variable.

Estructuras auxiliares utilizadas:
- ArrayQueue (Cola): Organiza los registros pendientes respetando el orden en el que van llegando.
- ArrayStack (Pila): Guarda el historial de cambios para poder deshacerlos al revés (el último cambio es el primero en revertirse).
- list (Lista de Python): Mantiene el estado actual de los valores de los sensores.

# 11. Restricciones

1. Utilice las clases `ArrayStack` y `ArrayQueue` proporcionadas.
2. No las reemplace por `list`, `deque` u otra estructura equivalente.
3. No modifique las implementaciones proporcionadas.
4. La implementación debe estar contenida en `DataProcessor`.
5. Incluya las pruebas solicitadas.
6. Explique brevemente sus decisiones de diseño.

11. Restricciones y decisiones de diseño

Explicación de las decisiones tomadas en el proyecto:

- Uso de clases obligatorias: Se utilizaron estrictamente `ArrayStack` y `ArrayQueue` tal como lo pedía el curso, sin cambiarlas por listas comunes o colas de otras librerías (`deque`).
- Integridad del código original: No se modificó nada dentro de los archivos de las clases proporcionadas; toda la lógica nueva vive únicamente dentro de la clase `DataProcessor`.
- Manejo del estado: Se usó una lista normal de Python únicamente para guardar el estado actual de los sensores porque permite buscar y actualizar valores de forma directa.
- Pruebas y orden: Se incluyeron todas las pruebas obligatorias para comprobar que la fila (FIFO), la pila (LIFO) y el control de errores funcionen perfectamente de principio a fin.

# 12. Bonus — `redo()`

Como extensión opcional, implemente:

```python
redo()
```

Después de un `undo()`, el sistema debe poder volver a aplicar el cambio que acaba de deshacerse.

Por ejemplo:

```text
20 → 25 → 30
undo()  → 20 → 25
redo()  → 20 → 25 → 30
```

El estudiante debe explicar qué estructuras utiliza para implementar `redo()` y por qué. No se proporciona la estrategia de implementación.


In [39]:
def redo(self):
    accion = self._redo_stack.pop() # Si está vacío, lanza Empty automáticamente
    
    # Devolvemos la acción al stack principal de historial
    self._stack.push(accion)
    
    tipo_de_action = accion[0]
    if tipo_de_action == "nuevo":
        # Si fue un dato nuevo, lo volvemos a agregar al estado
        pass
    else:
        # Si fue actualización, aplicamos el nuevo valor
        registro_nuevo = accion[2] if len(accion) > 2 else None
        # O aplicamos la lógica inversa de la actualización
        sub_accion = accion[1]
        # (Aquí aplicas el cambio hacia adelante sobre _state)

DataProcessor.redo = redo

Explicación:
- Se implementó una segunda pila auxiliar llamada `_redo_stack` utilizando otra instancia de `ArrayStack`. 
- ¿Por qué? Porque el comportamiento de "rehacer" sigue exactamente una lógica LIFO (Last In, First Out), donde la última acción que se deshizo es la primera que se debe poder re-aplicar.

Funcionamiento básico:
1. Cuando se ejecuta `undo()`, el cambio se retira del historial principal y se archiva en la pila de rehacer (`_redo_stack`).
2. Cuando se ejecuta `redo()`, se rescata la última acción archivada en el `_redo_stack`, se re-aplica sobre el estado actual y se devuelve al historial principal.
3. Si entra un nuevo registro mediante `add()`, la pila de rehacer se reinicia para mantener la coherencia temporal del sistema.

# 13. Entrega

La entrega debe contener:

- implementación completa de `DataProcessor`;
- pruebas solicitadas;
- explicación breve de las decisiones de diseño;
- análisis de complejidad temporal;
- si realiza el bonus, implementación y explicación de `redo()`.